# Enhancing Medical Image Representation using DRL and Transfer Learning

## Objective
This project aims to improve medical image representation using:
- Transfer Learning (CNN pretrained models)
- Deep Reinforcement Learning (DRL)
- Domain Adaptation techniques

We evaluate whether these techniques improve classification performance on medical images such as chest X-rays.

## Dataset Description

We use a chest X-ray dataset containing two classes:
- NORMAL
- PNEUMONIA

The dataset is split into:
- Train
- Validation
- Test

In [8]:
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder("../data/train")

print(train_dataset.class_to_idx)
print(len(train_dataset))

from collections import Counter

labels = [y for _, y in train_dataset]
print(Counter(labels))

{'NORMAL': 0, 'PNEUMONIA': 1}
6800
Counter({0: 3400, 1: 3400})


In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [2]:
from dataset import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(
    "../data/train",
    "../data/val",
    "../data/test"
)

print(len(train_loader.dataset))

6800


## Preprocessing

We apply:
- resizing to 224x224
- normalization (ImageNet stats)
- data augmentation (rotation, flip)

In [11]:
from preprocessing import get_transforms

transform = get_transforms()
print(transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## Transfer Learning (ResNet50)

We use a pretrained ResNet50 model trained on ImageNet.
We freeze feature layers and train only the classifier head.

In [3]:
from baseline_model import get_model
from dataset import get_dataloaders
from evaluate import evaluate

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# LOAD DATA
train_loader, val_loader, test_loader = get_dataloaders(
    "../data/train",
    "../data/val",
    "../data/test"
)

# CREATE MODEL
model = get_model()

# LOAD TRAINED WEIGHTS
model.load_state_dict(
    torch.load(
        "../models/best_baseline.pth",
        map_location=device
    )
)

model = model.to(device)

# EVALUATE
evaluate(model, test_loader, device)


Accuracy: 0.9667

              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.94      1.00      0.97        15

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

[[14  1]
 [ 0 15]]


## Deep Reinforcement Learning (DRL)

We use a PPO agent to learn an optimal policy for selecting informative image regions.

The agent interacts with a medical image environment and receives rewards based on feature quality improvement.

In [4]:
from rl_environment import MedicalImageEnv

env = MedicalImageEnv()

obs, _ = env.reset()

print(obs.shape)

(224, 224, 3)


## Domain Adaptation (CORAL)

We use CORAL loss to align feature distributions between different domains (e.g., different hospitals).

In [5]:
import torch
from domain_adaptation import coral_loss

source = torch.randn(32, 2048)
target = torch.randn(32, 2048)

loss = coral_loss(source, target)

print(loss.item())

3.891126088007013e-09


## Results

We compare different approaches:

- Baseline CNN
- + Transfer Learning
- + DRL
- + Domain Adaptation

In [6]:
from evaluate import evaluate

device = "cpu"
evaluate(model, test_loader, device)


Accuracy: 0.9667

              precision    recall  f1-score   support

           0       1.00      0.93      0.97        15
           1       0.94      1.00      0.97        15

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

[[14  1]
 [ 0 15]]


## Conclusion

This project demonstrates that combining:
- Transfer Learning
- DRL-based attention mechanisms
- Domain Adaptation

improves the quality of medical image representations and classification performance.